In [2]:
import pandas as pd 

def build_rolling_window_cohort(date_suffix="27Mar2026", window_months=24, fuzzy_tolerance=3):
    print(f"Building Rolling-Window Cohort ({window_months}-mo horizon, {fuzzy_tolerance}-mo tolerance)...")
    
    # --- [Data Loading & Renaming as before] ---
    df_my = pd.read_csv(f'../data/adni/raw_data/All_Subjects_My_Table_{date_suffix}.csv', low_memory=False)
    df_dx = pd.read_csv(f'../data/adni/raw_data/All_Subjects_DXSUM_{date_suffix}.csv', low_memory=False)
    df_demo = pd.read_csv(f'../data/adni/raw_data/All_Subjects_PTDEMOG_{date_suffix}.csv', low_memory=False)
    df_adas = pd.read_csv(f'../data/adni/raw_data/All_Subjects_ADAS_{date_suffix}.csv', low_memory=False)
    
    for df in [df_dx, df_demo, df_adas]:
        df.rename(columns={'VISCODE': 'visit', 'PTID': 'subject_id'}, inplace=True)

    def get_month(v):
        v = str(v).lower().strip()
        if v in ['bl', 'sc']: return 0
        if v.startswith('m'):
            try: return int(v.replace('m', ''))
            except: return -1
        return -1

    df_my['month'] = df_my['visit'].apply(get_month)
    df_dx['month'] = df_dx['visit'].apply(get_month)
    df_adas['month'] = df_adas['visit'].apply(get_month)
    
    # 2. Get Static Demographics from PTDEMOG (Gender, Education)
    print("Extracting static demographics from PTDEMOG...")
    bl_demo = df_demo.sort_values(['subject_id', 'visit']).groupby('subject_id').first().reset_index()
    demo_cols = ['subject_id', 'PTGENDER', 'PTEDUCAT']
    bl_demo = bl_demo[[c for c in demo_cols if c in bl_demo.columns]]

    # 3. Fuse Longitudinal Features (including Age and APOE from My_Table)
    print("Fusing longitudinal features and genetic data...")
    
    # Identify which columns are actually in My_Table
    # We look for Age (entry_age) and APOE4 here
    my_cols = ['subject_id', 'visit', 'month', 'entry_research_group', 'entry_age', 'MMSCORE', 'CDRSB', 'FAQTOTAL', 'MOCA', 'NPISCORE', 'GENOTYPE']

    
    long_features = df_my[my_cols].copy()

    long_features['age_at_visit'] = long_features['entry_age'] + (long_features['month'] / 12)

    
    long_adas = df_adas[['subject_id', 'month', 'TOTAL13']].drop_duplicates(subset=['subject_id', 'month'])
    long_features = long_features.merge(long_adas, on=['subject_id', 'month'], how='left')
    


    # Merge Gender and Education from the PTDEMOG subset we made
    long_features = long_features.merge(bl_demo, on='subject_id', how='left')

    # ---------------------------------------------------------
    # CONSOLIDATE SC/BL (MONTH 0)
    # ---------------------------------------------------------
    print("Consolidating 'sc' and 'bl' visits to maximize Baseline features...")
    
    # Define clinical columns that might be split between sc and bl
    clin_cols = my_cols[4:]  # Assuming first 4 cols are subject_id, visit, month, entry_research_group
    
    # 1. Sort to ensure 'sc' (screening) is processed before 'bl' (baseline)
    long_features = long_features.sort_values(['subject_id', 'month', 'visit'])
    
    # 2. Within each patient's Month 0, fill NAs using both rows
    # (e.g., if MMSE is in 'sc' and ADAS is in 'bl', the 'bl' row will now have both)
    m0_mask = long_features['month'] == 0
    long_features.loc[m0_mask, clin_cols] = long_features[m0_mask].groupby('subject_id')[clin_cols].ffill().bfill()
    
    # 3. Drop the 'sc' row, keeping the 'bl' row which now has the combined data
    # If they have both, we keep 'last' (usually the 'bl' visit)
    long_features = long_features.drop_duplicates(subset=['subject_id', 'month'], keep='last')
    
    # ---------------------------------------------------------
    # --- [Label Generation with Diagnosis Gatekeeper as before] ---
    valid_rows = []
    dx_grouped = dict(tuple(df_dx.groupby('subject_id')))
    
    for idx, row in long_features.iterrows():
        subj = row['subject_id']
        current_month = row['month']
        if current_month < 0 or subj not in dx_grouped: continue
        
        patient_dx = dx_grouped[subj]
        
        # ---------------------------------------------------------
        # DYNAMIC GATEKEEPER: Strict (0) or Fuzzy (>0)
        # ---------------------------------------------------------
        if fuzzy_tolerance == 0:
            # Strict mode: Exact match only
            current_dx_row = patient_dx[patient_dx['month'] == current_month]
        else:
            # Fuzzy mode: Look for nearest record within the tolerance window
            window = patient_dx[
                (patient_dx['month'] >= current_month - fuzzy_tolerance) & 
                (patient_dx['month'] <= current_month + fuzzy_tolerance)
            ].copy()
            
            if not window.empty:
                window['dist'] = (window['month'] - current_month).abs()
                current_dx_row = window.sort_values('dist').head(1)
            else:
                current_dx_row = pd.DataFrame() # Empty if nothing in window

        # --- Check if they are MCI at this point ---
        is_mci_now = False
        if not current_dx_row.empty:
            target_row = current_dx_row.iloc[0]
            # Harmonized ADNI check: 2 = MCI
            if 'DIAGNOSIS' in target_row.index and target_row['DIAGNOSIS'] == 2:
                is_mci_now = True
        
        if not is_mci_now: continue
            
        # ---------------------------------------------------------
        # LABELING (Window-based)
        # ---------------------------------------------------------
        target_month = current_month + window_months
        future_window = patient_dx[(patient_dx['month'] > current_month) & (patient_dx['month'] <= target_month)]
        
        # Harmonized ADNI check: 3 = Dementia/AD
        if 'DIAGNOSIS' in future_window.columns and 3 in future_window['DIAGNOSIS'].values: 
            row['label'] = 1
            
            # --- NEW: Track exact distance to conversion ---
            conversion_rows = future_window[future_window['DIAGNOSIS'] == 3]
            first_conversion_month = conversion_rows['month'].min()
            row['months_to_conversion'] = first_conversion_month - current_month
            
            valid_rows.append(row)
        else:
            # Stable: Must have stayed in study for at least (Window - Tolerance)
            if patient_dx['month'].max() >= (target_month - fuzzy_tolerance):
                row['label'] = 0
                row['months_to_conversion'] = -1  # Use -1 to represent Stable
                valid_rows.append(row)

    final_df = pd.DataFrame(valid_rows)
    def encode_apoe(genotype):
        genotype = str(genotype)
        if '4/4' in genotype: return 2
        if '4' in genotype: return 1
        return 0

    final_df['APOE4_count'] = final_df['GENOTYPE'].apply(encode_apoe)
    # Drop the original string column
    final_df.drop(columns=['GENOTYPE'], inplace=True, errors='ignore')

    
    
    # ---------------------------------------------------------
    # NEW: VISIT COUNT AUDIT
    # ---------------------------------------------------------
    print("\n" + "="*50)
    print("LONGITUDINAL VISIT AUDIT")
    print("="*50)
    
    visit_counts = final_df['subject_id'].value_counts()
    
    print(f"Total Training Rows:      {len(final_df)}")
    print(f"Unique Patients:          {len(visit_counts)}")
    print(f"Average Rows per Patient: {visit_counts.mean():.2f}")
    print(f"Max Rows from one Patient: {visit_counts.max()}")
    print(f"Min Rows from one Patient: {visit_counts.min()}")
    
    print("\n--- Rows per Patient Distribution ---")
    dist = visit_counts.value_counts().sort_index()
    print(f"{'Rows':<10} | {'Number of Patients'}")
    print("-" * 30)
    for num_rows, num_patients in dist.items():
        print(f"{num_rows:<10} | {num_patients}")
        
    print("\n" + "="*50)
    print(f"Prior (Prevalence): {round(final_df['label'].mean(), 3)}")
    print("="*50)
    
    final_df.set_index(['subject_id', 'month'], inplace=True)
    return final_df

window_months = 36
rolling_dataset = build_rolling_window_cohort(date_suffix="27Mar2026", window_months=window_months)

Building Rolling-Window Cohort (36-mo horizon, 3-mo tolerance)...
Extracting static demographics from PTDEMOG...
Fusing longitudinal features and genetic data...
Consolidating 'sc' and 'bl' visits to maximize Baseline features...

LONGITUDINAL VISIT AUDIT
Total Training Rows:      883
Unique Patients:          316
Average Rows per Patient: 2.79
Max Rows from one Patient: 6
Min Rows from one Patient: 1

--- Rows per Patient Distribution ---
Rows       | Number of Patients
------------------------------
1          | 92
2          | 51
3          | 78
4          | 37
5          | 41
6          | 17

Prior (Prevalence): 0.655


In [3]:
rolling_dataset

visit entry_research_group  entry_age  MMSCORE  CDRSB  \
subject_id month                                                         
002_S_0729 0        sc                  MCI      65.18     27.0    0.5   
           6       m06                  MCI      65.18     27.0    0.5   
002_S_0782 0        sc                  MCI      81.65     29.0    0.5   
002_S_0954 0        sc                  MCI      69.40     25.0    2.0   
           6       m06                  MCI      69.40     24.0    1.5   
...                ...                  ...        ...      ...    ...   
941_S_1295 6       m06                  MCI      76.75     25.0    1.5   
           12      m12                  MCI      76.75     24.0    2.5   
941_S_1311 0        sc                  MCI      69.14     29.0    2.5   
           6       m06                  MCI      69.14     27.0    3.5   
941_S_1363 0        sc                  MCI      69.82     24.0    3.0   

                  FAQTOTAL  MOCA  NPISCORE  age_at_visit  TOTAL13  PTGENDER  \
subject_id month                                                              
002_S_0729 0           7.0  21.0       0.0         65.18    14.67       2.0   
           6           5.0   NaN       0.0         65.68    17.00       2.0   
002_S_0782 0           0.0  21.0       1.0         81.65    23.33       1.0   
002_S_0954 0           1.0  21.0       0.0         69.40    21.67       2.0   
           6           2.0   NaN       2.0         69.90    25.33       2.0   
...                    ...   ...       ...           ...      ...       ...   
941_S_1295 6           1.0   NaN       1.0         77.25    20.00       1.0   
           12          1.0   NaN       3.0         77.75    26.00       1.0   
941_S_1311 0           8.0  19.0       3.0         69.14    18.33       1.0   
           6          16.0   NaN       3.0         69.64    17.00       1.0   
941_S_1363 0           7.0  19.0       3.0         69.82    27.33       2.0   

                  PTEDUCAT  label  months_to_conversion  APOE4_count  
subject_id month                                                      
002_S_0729 0          16.0      1                    12            1  
           6          16.0      1                     6            1  
002_S_0782 0          16.0      0                    -1            0  
002_S_0954 0          14.0      1                    12            1  
           6          14.0      1                     6            1  
...                    ...    ...                   ...          ...  
941_S_1295 6          16.0      1                    12            1  
           12         16.0      1                     6            1  
941_S_1311 0          12.0      1                    12            2  
           6          12.0      1                     6            2  
941_S_1363 0          12.0      1                     6            1  

[883 rows x 15 columns]

APPLY LOCF

In [29]:
def prep_and_save_data(rolling_df):
    df = rolling_df.copy()
    
    print("Applying patient-specific forward-fill (LOCF)...")
    # Sort chronologically so forward-fill works correctly
    df = df.sort_values(['subject_id', 'month'])
    
    # Safely impute longitudinally 
    clinical_scores = ['TOTAL13', 'MMSCORE', 'CDRSB', 'FAQTOTAL', 'MOCA']
    df[clinical_scores] = df.groupby('subject_id')[clinical_scores].ffill()
    
    # Format Gender
    if df['PTGENDER'].dtype == 'O':
        df['PTGENDER'] = df['PTGENDER'].map({'Male': 0, 'Female': 1, 'M': 0, 'F': 1})
        df['PTGENDER'] = pd.to_numeric(df['PTGENDER'], errors='coerce').fillna(0)
  
    
    return df

# --- RUN IT ---
final_df = prep_and_save_data(rolling_dataset)

Applying patient-specific forward-fill (LOCF)...


In [30]:
cols_to_drop = [
    'month',                # Use age_at_visit instead to avoid study-duration leakage
    'visit',                # Metadata only
    'entry_age',            # Redundant with age_at_visit
    'AGE',                  # Redundant with age_at_visit
    'entry_research_group'  # We already filtered for MCI; this is now administrative noise
]

final_features_df = final_df.drop(columns=[c for c in cols_to_drop if c in rolling_dataset.columns])

print(f"Post-cleanup columns: {final_features_df.columns.tolist()}")

Post-cleanup columns: ['MMSCORE', 'CDRSB', 'FAQTOTAL', 'MOCA', 'NPISCORE', 'age_at_visit', 'TOTAL13', 'PTGENDER', 'PTEDUCAT', 'label', 'months_to_conversion', 'APOE4_count']


In [31]:
final_df

visit entry_research_group  entry_age  MMSCORE  CDRSB  \
subject_id month                                                         
002_S_0729 0        sc                  MCI      65.18     27.0    0.5   
           6       m06                  MCI      65.18     27.0    0.5   
002_S_0782 0        sc                  MCI      81.65     29.0    0.5   
002_S_0954 0        sc                  MCI      69.40     25.0    2.0   
           6       m06                  MCI      69.40     24.0    1.5   
...                ...                  ...        ...      ...    ...   
941_S_1295 6       m06                  MCI      76.75     25.0    1.5   
           12      m12                  MCI      76.75     24.0    2.5   
941_S_1311 0        sc                  MCI      69.14     29.0    2.5   
           6       m06                  MCI      69.14     27.0    3.5   
941_S_1363 0        sc                  MCI      69.82     24.0    3.0   

                  FAQTOTAL  MOCA  NPISCORE  age_at_visit  TOTAL13  PTGENDER  \
subject_id month                                                              
002_S_0729 0           7.0  21.0       0.0         65.18    14.67       2.0   
           6           5.0  21.0       0.0         65.68    17.00       2.0   
002_S_0782 0           0.0  21.0       1.0         81.65    23.33       1.0   
002_S_0954 0           1.0  21.0       0.0         69.40    21.67       2.0   
           6           2.0  21.0       2.0         69.90    25.33       2.0   
...                    ...   ...       ...           ...      ...       ...   
941_S_1295 6           1.0  19.0       1.0         77.25    20.00       1.0   
           12          1.0  19.0       3.0         77.75    26.00       1.0   
941_S_1311 0           8.0  19.0       3.0         69.14    18.33       1.0   
           6          16.0  19.0       3.0         69.64    17.00       1.0   
941_S_1363 0           7.0  19.0       3.0         69.82    27.33       2.0   

                  PTEDUCAT  label  months_to_conversion  APOE4_count  
subject_id month                                                      
002_S_0729 0          16.0      1                    12            1  
           6          16.0      1                     6            1  
002_S_0782 0          16.0      0                    -1            0  
002_S_0954 0          14.0      1                    12            1  
           6          14.0      1                     6            1  
...                    ...    ...                   ...          ...  
941_S_1295 6          16.0      1                    12            1  
           12         16.0      1                     6            1  
941_S_1311 0          12.0      1                    12            2  
           6          12.0      1                     6            2  
941_S_1363 0          12.0      1                     6            1  

[883 rows x 15 columns]

In [32]:
final_features_df[[ 'MMSCORE', 'CDRSB', 'FAQTOTAL', 'MOCA', 
                   'NPISCORE', 'age_at_visit', 'TOTAL13', 'PTGENDER', 'PTEDUCAT', 
                   'APOE4_count', 'label', 'months_to_conversion']].to_csv(f"../data/adni/adni_rolling_locf_{window_months}m.csv", index=True)

Extract only baseline visits and store the baseline dataset

In [33]:
baseline_dataset = rolling_dataset.xs(0, level='month', drop_level=False)
baseline_dataset[['MMSCORE', 'CDRSB', 'FAQTOTAL', 'MOCA', 
                   'NPISCORE', 'age_at_visit', 'TOTAL13', 'PTGENDER', 'PTEDUCAT', 
                   'APOE4_count', 'label', 'months_to_conversion']].to_csv(f"../data/adni/adni_baseline_only_{window_months}m.csv", index=True)